In [4]:
import pandas as pd
import requests
import time
import os

In [12]:
class DeleteEmbbedings:
    def __init__(self, arquivo_excel=None):
        self.ARQUIVO_EXCEL = (
            arquivo_excel
            or r"C:\Users\pmarcelino\OneDrive - Indra\Documentos\Aotomações\Vivo\Scripts\ScriptEmmbbeding\planilha.csv"
        )
        self.df = self.read_csv()
        self.headers = self.build_headers()

    def read_csv(self):
        return pd.read_csv(self.ARQUIVO_EXCEL, sep=",", encoding="utf-8")

    def _load_env_values(self):
        from pathlib import Path

        for base_path in [Path.cwd(), *Path.cwd().parents]:
            env_path = base_path / ".env"
            if not env_path.exists():
                continue

            values = {}
            for line in env_path.read_text(encoding="utf-8").splitlines():
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue

                key, value = line.split("=", 1)
                values[key.strip()] = value.strip().strip('"').strip("'")

            return values

        return {}

    def build_headers(self):
        env_values = self._load_env_values()

        token = os.getenv("EMBEDDINGS_TOKEN") or env_values.get("TOKEN", "")
        session_id = os.getenv("EMBEDDINGS_SESSION_ID") or env_values.get(
            "EMBEDDINGS_SESSION_ID", "af0663be-c916-4e53-ae0c-544e73f19aa1"
        )
        subscription_id = os.getenv("EMBEDDINGS_SUBSCRIPTION_ID") or env_values.get(
            "EMBEDDINGS_SUBSCRIPTION_ID", "6495aca06e4f0f95368f289c"
        )
        subscription_name = os.getenv("EMBEDDINGS_SUBSCRIPTION_NAME") or env_values.get(
            "EMBEDDINGS_SUBSCRIPTION_NAME", "Bull Dashboard"
        )
        cookie = os.getenv("EMBEDDINGS_COOKIE") or env_values.get("COOKIE", "")

        base_headers = {
            "accept": "application/json, text/plain, */*",
            "accept-language": "en-US,en;q=0.9",
            "oam_ccusto": "nao_encontrado",
            "oam_gestor": "nao_encontrado",
            "oam_matricula": "nao_encontrado",
            "priority": "u=1, i",
            "sec-ch-ua-mobile": "?0",
            "sec-ch-ua-platform": '"Windows"',
            "sec-fetch-dest": "empty",
            "sec-fetch-mode": "cors",
            "sec-fetch-site": "same-origin",
            "service": "QNA_MOBILE_PRODUCTS",
        }

        auth_headers = {
            "session_id": session_id,
            "subscription_name": subscription_name,
            "token": token,
            "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36 Edg/147.0.0.0",
            "user_email": "diego.souza@telefonica.com",
            "user_id": "A0000000",
            "user_name": "Diego Magalhaes De Souza",
            "user_profile": "VGPT_ADMIN_LAB_IA",
            "user_profile_permissions": "READ,CREATE,UPDATE,DELETE",
            "Cookie": cookie,
        }

        request_headers = {
            "origin": "https://ai-br-pre.telefonicabigdata.com",
            "referer": f"https://ai-br-pre.telefonicabigdata.com/lib/orchestrator?subscription_id={subscription_id}",
            "subscription_id": subscription_id,
        }

        return {**base_headers, **auth_headers, **request_headers}

    def _mark_as_processed(self, index):
        self.df.iloc[index, 1] = "Sim"
        self.df.to_csv(self.ARQUIVO_EXCEL, index=False)

    def _process_row(self, index, row):
        nome_item = row.iloc[0]
        processado = row.iloc[1]

        if str(processado).lower() == "sim":
            print(f"Pulou linha {index}: {nome_item}")
            return

        try:
            url = f"https://ai-br-pro.telefonicabigdata.com/pdf/{nome_item}"
            print(f"Removendo item: {nome_item}")

            response = requests.delete(
                url=url, headers=self.headers, timeout=30, verify=False
            )

            if response.status_code == 204:
                print(f"linha {index}. Item {nome_item} removido com sucesso")
                self._mark_as_processed(index)
            else:
                print(f"Erro ao remover {nome_item} - Status: {response.status_code}")
                print(response.text)

        except Exception as exc:
            print(f"Erro ao processar {nome_item}: {exc}")

        time.sleep(0.5)

    def run(self):
        for index, row in self.df.iterrows():
            self._process_row(index, row)

        print("Processamento finalizado.")


# Exemplo de uso:
processor = DeleteEmbbedings()
processor.run()

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\pmarcelino\\OneDrive - Indra\\Documentos\\Aotomações\\Vivo\\Scripts\\ScriptEmmbbeding\\planilha.csv'